# Patient Allocation for Disease Modeling

**Purpose**: Eliminate double/triple counting in overlapping HSAs for accurate disease rate calculations

## Why This Step is Necessary

HSAs created by `HSA_v6_FINAL.ipynb` are **overlapping circular regions** around facilities:
- Each facility has a service radius (10-25 km)
- Service areas overlap where circles intersect
- Same population pixel can be in 2-3 overlapping HSAs

**Problem for disease modeling**:
- Without allocation: same person counted multiple times in disease rate denominators
- Example: Person living in overlap zone counted in HSA_A, HSA_B, and HSA_C
- Result: Inflated population totals, incorrect disease rates

**Solution**:
- Gravity model assigns each population pixel to **exactly ONE facility**
- Based on distance + facility size (larger facilities attract from farther away)
- Result: Non-overlapping population assignments, accurate denominators

**Computational Optimization**:
- Uses sampling (default: every 10th pixel) for faster processing
- Allocated populations are scaled up by sample rate to represent full population
- Maintains statistical accuracy while reducing computation time

## When to Use This Notebook

**Required if**:
- Computing disease rates by HSA
- Aggregating weekly disease counts by HSA
- Building predictive models with HSA-level data

**Not required if**:
- Only visualizing HSA boundaries
- Spatial analysis without disease rates
- Descriptive statistics about facility coverage

---

## Inputs Required

1. **HSA boundaries**: GeoJSON from `HSA_v6_FINAL.ipynb` (select which mode: FEWEST, FOOTPRINT, DISTANCE, etc.)
2. **Population raster**: `data/jor_ppp_2020_UNadj.tif`
3. **Facility data**: CSV with coordinates and patient volumes

## Outputs

1. **HSA population summary**: Total allocated population per HSA (no double counting, scaled to full population)
2. **CSV file**: `out/hsa_allocated_patients_{mode}.csv` for use in disease modeling
3. **Pixel allocations**: `out/pixel_allocations_{mode}.csv` (optional, at sampled scale for reference)

In [ ]:
%load_ext autoreload
%autoreload 2
# ============================================================================
# CONFIGURATION
# ============================================================================

from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np
from patient_allocation import PatientAllocator

# Directories
DATA_DIR = Path("data")
OUT_DIR = Path("out")

# Network type
NETWORK = "NCD"  # Options: 'INF' (infectious disease) or 'NCD' (non-communicable disease). Also, SYNINF or SYNNCD for synthetic datasets

# ============================================================================
# SELECT HSA MODE FOR MODELING
# ============================================================================
# Choose which HSA optimization result to use for disease modeling.
# Available modes from HSA_v6_FINAL.ipynb:
#   - 'fewest': Minimum number of HSAs (typically 14-15)
#   - 'footprint': Maximum geographic diversity (typically 18-19)
#   - 'distance': Minimize travel distance (typically 15-17)
#   - 'governorate_tau_coverage': 90% coverage per governorate (typically 16)
#   - 'governorate_fewest': One anchor per governorate + minimize total (typically 16)

HSA_MODE = "footprint"  # ← CHANGE THIS to select your HSA mode

# Population raster
POP_RASTER = DATA_DIR / "jor_ppp_2020_UNadj.tif"

# ============================================================================
# GRAVITY MODEL PARAMETERS
# ============================================================================
# IMPORTANT: For production use, set sample_rate = 1 to process ALL pixels
# Use sample_rate > 1 only for quick testing (results will be scaled up)
# ============================================================================

GRAVITY_PARAMS = {
    'alpha': 0.75,          # Facility size weight (higher = facility volume matters more)
    'beta': 1.5,            # Distance decay exponent (higher = distance matters more)
    'max_distance_km': 200, # Maximum travel distance considered (increased from 100)
    'sample_rate': 1        # ← IMPORTANT: Set to 1 for production, >1 for testing
}

print("="*80)
print("PATIENT ALLOCATION CONFIGURATION")
print("="*80)
print(f"Network: {NETWORK}")
print(f"HSA Mode: {HSA_MODE}")
print(f"Population Raster: {POP_RASTER.name}")
print(f"Gravity Model Parameters:")
for key, val in GRAVITY_PARAMS.items():
    print(f"  {key}: {val}")
if GRAVITY_PARAMS['sample_rate'] > 1:
    print("\n⚠️  WARNING: sample_rate > 1 (using sampling for speed)")
    print("   Results will be scaled up. For production use, set sample_rate = 1")
else:
    print("\n✓ Processing all pixels (sample_rate = 1) - production mode")
print("="*80)

## 1. Load HSA Boundaries and Facilities

In [ ]:
# Load HSA boundaries (output from HSA_v6_FINAL.ipynb)
hsa_file = OUT_DIR / f"{NETWORK}_{HSA_MODE}_hsas_v2.geojson"

if not hsa_file.exists():
    raise FileNotFoundError(
        f"HSA file not found: {hsa_file}\n"
        f"Please run HSA_v6_FINAL.ipynb first to generate HSA boundaries for mode '{HSA_MODE}'."
    )

hsas = gpd.read_file(hsa_file)
print(f"Loaded {len(hsas)} HSAs from {hsa_file.name}")
print(f"  CRS: {hsas.crs}")
print(f"  Columns: {list(hsas.columns)}")

# Display HSA summary
print("\nHSA Anchor Facilities:")
for idx, row in hsas.iterrows():
    fac_name = row.get('HealthFacility', row.get('FacilityName', f'Facility_{idx}'))
    radius = row.get('service_radius_km', row.get('initial_radius_km', 'N/A'))
    print(f"  {idx+1}. {fac_name} (radius: {radius} km)")

In [ ]:
# Load ALL facilities (not just HSA anchors)
# Patient allocation considers all facilities in the gravity model

facilities_csv = OUT_DIR / f"{NETWORK}_Facilities_Climate_Features_with_clusters.csv"

if not facilities_csv.exists():
    raise FileNotFoundError(
        f"Facilities file not found: {facilities_csv}\n"
        f"Please ensure the climate features CSV is in the data directory."
    )

facilities_df = pd.read_csv(facilities_csv)

# Normalize whitespace in facility names
if 'FacilityName' in facilities_df.columns:
    facilities_df['HealthFacility'] = facilities_df['FacilityName'].str.replace(r'\s+', ' ', regex=True).str.strip()
elif 'HealthFacility' in facilities_df.columns:
    facilities_df['HealthFacility'] = facilities_df['HealthFacility'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Load diagnosis counts for patient volumes
diagnosis_csv = OUT_DIR / f"{NETWORK}_diagnosis_counts_pivot.csv"

if diagnosis_csv.exists():
    diagnosis_df = pd.read_csv(diagnosis_csv)
    diagnosis_df['healthfacility'] = diagnosis_df['healthfacility'].str.replace(r'\s+', ' ', regex=True).str.strip()
    
    # Merge to get Total column
    facilities_df = facilities_df.merge(
        diagnosis_df[['healthfacility', 'total_diagnoses']],
        left_on='HealthFacility',
        right_on='healthfacility',
        how='left'
    )
    facilities_df['Total'] = facilities_df['total_diagnoses'].fillna(0)
else:
    print(f"WARNING: Diagnosis counts not found at {diagnosis_csv}")
    print("  Using uniform facility weights (Total = 1000 for all facilities)")
    facilities_df['Total'] = 1000  # Uniform weight if no patient data

# Create GeoDataFrame
facilities = gpd.GeoDataFrame(
    facilities_df,
    geometry=gpd.points_from_xy(facilities_df['lon'], facilities_df['lat']),
    crs='EPSG:4326'
)

print(f"\nLoaded {len(facilities)} facilities from {facilities_csv.name}")
print(f"  Facilities with patient volumes > 0: {(facilities['Total'] > 0).sum()}")
print(f"  Total patient volume across all facilities: {facilities['Total'].sum():,.0f}")

## 2. Run Patient Allocation

This step assigns each population pixel to exactly one facility using a gravity model:

$$
\text{Attractiveness}(f) = \frac{\text{Volume}(f)^\alpha}{\text{Distance}(f)^\beta}
$$

Where:
- $\alpha$ = facility size weight (higher values favor larger facilities)
- $\beta$ = distance decay exponent (higher values favor closer facilities)

Each pixel is assigned to the facility with maximum attractiveness.

In [ ]:
# Initialize patient allocator
print("Initializing PatientAllocator...")
allocator = PatientAllocator(
    pop_raster_path=str(POP_RASTER),
    facilities_gdf=facilities,
    params=GRAVITY_PARAMS
)

print(f"  Population raster: {POP_RASTER.name}")
print(f"  Facilities: {len(facilities)}")
print(f"  Alpha (facility size weight): {GRAVITY_PARAMS['alpha']}")
print(f"  Beta (distance decay): {GRAVITY_PARAMS['beta']}")
print(f"  Max travel distance: {GRAVITY_PARAMS['max_distance_km']} km")

In [ ]:
# Allocate all population pixels to facilities
print("\nAllocating population pixels to facilities...")
print("  (This may take several minutes for full resolution raster)")

allocated = allocator.allocate_all_pixels_parallel()

print("\nAllocation complete!")
print(f"  Total pixels processed: {len(allocated):,}")
print(f"  Pixels with population > 0: {(allocated['population'] > 0).sum():,}")
print(f"  Total population allocated: {allocated['population'].sum():,.0f}")
print(f"  Facilities with allocated population: {allocated['facility_id'].nunique()}")

## 3. Aggregate by HSA

Sum allocated populations by HSA anchor facility to get non-overlapping population totals.

In [ ]:
# Aggregate allocated populations by HSA
print("Aggregating by HSA...")
hsa_populations = allocator.aggregate_by_hsa(allocated, hsas)

# Scale up to account for sampling (we processed every Nth pixel)
sample_rate = GRAVITY_PARAMS['sample_rate']
print(f"Scaling populations by sample rate: × {sample_rate}")
hsa_populations['allocated_patients'] = hsa_populations['allocated_patients'] * sample_rate

print(f"\nHSA Population Summary (mode: {HSA_MODE}):")
print("="*80)
print(hsa_populations.to_string(index=False))
print("="*80)

# Summary statistics
print(f"\nTotal HSAs: {len(hsa_populations)}")
print(f"Total allocated population: {hsa_populations['allocated_patients'].sum():,.0f}")
print(f"Mean population per HSA: {hsa_populations['allocated_patients'].mean():,.0f}")
print(f"Min population: {hsa_populations['allocated_patients'].min():,.0f}")
print(f"Max population: {hsa_populations['allocated_patients'].max():,.0f}")

## 4. Save Results for Disease Modeling

In [ ]:
# Save allocated populations to CSV (already scaled in cell-9)
output_csv = OUT_DIR / f"hsa_allocated_patients_{NETWORK}_{HSA_MODE}.csv"
hsa_populations.to_csv(output_csv, index=False)

print(f"Saved allocated populations to: {output_csv}")
print(f"  (Populations already scaled by sample rate: × {GRAVITY_PARAMS['sample_rate']})")
print(f"\nFile size: {output_csv.stat().st_size / 1024:.1f} KB")

# Also save the full allocated pixel data (optional, for detailed analysis)
# Note: Pixel data remains at sampled scale for reference
allocated_raster_csv = OUT_DIR / f"pixel_allocations_{NETWORK}_{HSA_MODE}.csv"
allocation_details_csv = OUT_DIR / f"{NETWORK}_{HSA_MODE}_allocation_details.csv"
allocated.to_csv(allocated_raster_csv, index=False)
allocated.to_csv(allocation_details_csv, index=False)
print(f"Saved allocation details to: {allocation_details_csv}")

print(f"Saved pixel-level allocations to: {allocated_raster_csv}")
print(f"  (Pixel data is at sampled scale for reference)")
print(f"File size: {allocated_raster_csv.stat().st_size / (1024*1024):.1f} MB")

print("\n" + "="*80)
print("PATIENT ALLOCATION COMPLETE")
print("="*80)
print(f"Use {output_csv.name} as denominators for disease rate calculations.")
print(f"Each HSA has a non-overlapping allocated population total (scaled to full population).")

## 5. Validation and Diagnostics

Check allocation quality and identify potential issues.

In [ ]:
# Compare allocated populations to original raster total
import rasterio

with rasterio.open(POP_RASTER) as src:
    pop_data = src.read(1, masked=True)
    total_pop_raster = float(pop_data.sum())

# Account for sampling: we processed every Nth pixel, so scale up the allocated population
sample_rate = GRAVITY_PARAMS['sample_rate']
total_pop_allocated_sampled = allocated['population'].sum()
total_pop_allocated_scaled = total_pop_allocated_sampled * sample_rate

print("Population Coverage:")
print(f"  Total population in raster: {total_pop_raster:,.0f}")
print(f"  Sample rate: 1/{sample_rate} (processing every {sample_rate}th pixel)")
print(f"  Sampled population allocated: {total_pop_allocated_sampled:,.0f}")
print(f"  Scaled population allocated: {total_pop_allocated_scaled:,.0f} (× {sample_rate})")

# Calculate coverage based on scaled allocated population
coverage_pct = (total_pop_allocated_scaled / total_pop_raster) * 100
print(f"  Coverage: {coverage_pct:.2f}%")

if coverage_pct < 95:
    print(f"\n  WARNING: Only {coverage_pct:.1f}% of population allocated.")
    print(f"  This may indicate max_distance_km is too low ({GRAVITY_PARAMS['max_distance_km']} km).")
    print(f"  Consider increasing max_distance_km parameter.")

# Check for HSAs with very low populations (potential issues)
# Note: Populations in hsa_populations are already scaled
low_pop_threshold = 100000  # Scaled threshold for full population
low_pop_hsas = hsa_populations[hsa_populations['allocated_patients'] < low_pop_threshold]

if len(low_pop_hsas) > 0:
    print(f"\nHSAs with allocated population < {low_pop_threshold:,}:")
    print(low_pop_hsas[['anchor_name', 'allocated_patients']].to_string(index=False))
    print("  (This is not necessarily a problem - may be rural/remote facilities)")

# Check distribution of allocated population across facilities
# Note: facility_pops are at sampled scale
print(f"\nPopulation allocation by facility (sampled scale):")
facility_pops = allocated.groupby('facility_id')['population'].sum().sort_values(ascending=False)
print(f"  Top 10 facilities by allocated population:")
for fac_id, pop in facility_pops.head(10).items():
    if fac_id in facilities.index:
        fac_name = facilities.loc[fac_id].get('HealthFacility', f'Facility_{fac_id}')
    else:
        match = facilities[facilities['HealthFacility'] == fac_id]
        fac_name = match.iloc[0]['HealthFacility'] if not match.empty else f'Facility_{fac_id}'
    # Scale up for display
    scaled_pop = pop * sample_rate
    print(f"    {fac_name}: {scaled_pop:,.0f}")

print("\n" + "="*80)
print("VALIDATION COMPLETE")
print("="*80)